# EN4554 Assignment 2 — Convolutional Neural Networks

| | |
|---|---|
| **Name** | Manujaya U.G.P. |
| **Index number** | 220386H |
| **GitHub repository** | https://github.com/Manujayaugp/EN4554-A02 |

**Use of AI tools.** I used Claude as a coding assistant; I understand all the code and can explain it.

### How to run
1. Open the notebook in Google Colab and choose **Runtime → Change runtime type → T4 GPU**.
2. Choose **Runtime → Run all**. With `QUICK = False`, the full notebook takes about 1.5–2.5 hours on a T4. Set `QUICK = True` first for a smoke test of every cell (a few minutes).
3. Long experiments are cached on Google Drive (`USE_DRIVE = True`; Colab asks for permission to mount Drive), so that a disconnected session can resume where it stopped. Set `USE_DRIVE = False` to cache in the local `results/` folder instead.

### Protocol used in every task
* The CIFAR-10 training set is split into **45 000 training / 5 000 validation** images with a fixed random permutation (seed 0).
* The **per-channel mean and std come from the 45k training split only** and are applied to the training, validation and test images.
* Random seeds are fixed with `set_seed`. Every run in a comparison starts from the same initial weights and sees the same mini-batch order.
* The **test set is used exactly once**, at the end of Task 4.

## 0. Setup

In [ ]:
import os, time, math, copy, json, random, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler
from IPython.display import display
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

QUICK = False        # True: tiny smoke test of every cell; False: the full experiments
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'   # fp16 mixed precision for the larger CNNs (Tasks 3 and 4)

print('PyTorch', torch.__version__, '| torchvision', torchvision.__version__)
print('Device :', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU')
if device.type != 'cuda':
    print('WARNING: no GPU. In Colab use Runtime > Change runtime type > T4 GPU.')

SEED = 0
def set_seed(seed=SEED):
    """Seed Python, NumPy and PyTorch (CPU and every GPU)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
set_seed()
torch.backends.cudnn.benchmark = True   # fastest cuDNN kernels (may cause tiny run-to-run differences)

# Plot style: one fixed colour order for all figures (colour-blind-safe for adjacent series).
PALETTE = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948']
plt.rcParams.update({
    'figure.dpi': 110, 'axes.prop_cycle': cycler(color=PALETTE), 'lines.linewidth': 2,
    'axes.grid': True, 'grid.color': '#e1e0d9', 'grid.linewidth': 0.8,
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': '#52514e', 'xtick.color': '#52514e',
    'ytick.color': '#52514e', 'axes.titleweight': 'bold', 'axes.spines.top': False,
    'axes.spines.right': False, 'legend.frameon': False})

# Results of every long experiment are cached as JSON so that a disconnected
# Colab session can resume.  Optionally keep the cache on Google Drive.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = '/content/drive/MyDrive/en4554_a02_results'
else:
    RESULTS_DIR = './results'
if QUICK:
    RESULTS_DIR += '_quick'
os.makedirs(RESULTS_DIR, exist_ok=True)

def cached(name, fn):
    """Run fn() once and store its JSON-serialisable result; reload it on later runs."""
    path = os.path.join(RESULTS_DIR, name + '.json')
    if os.path.exists(path):
        print(f'[cache] loaded {name}')
        with open(path) as f:
            return json.load(f)
    t0 = time.time()
    out = json.loads(json.dumps(fn()))      # same types whether computed or loaded
    with open(path, 'w') as f:
        json.dump(out, f)
    print(f'[{name}] finished in {(time.time() - t0) / 60:.1f} min')
    return out

### CIFAR-10: 45k / 5k split and per-channel normalisation
The whole dataset (about 0.6 GB as float32) fits in GPU memory. Mini-batches are therefore sliced directly on the GPU instead of using a `DataLoader`. This is much faster on Colab, which has only 2 CPU cores for image decoding and augmentation.

In [ ]:
DATA_DIR = './data'
cifar_train = torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True)
cifar_test = torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True)   # untouched until Task 4
CLASSES = cifar_train.classes

X_all = torch.from_numpy(cifar_train.data).permute(0, 3, 1, 2).float().div_(255)   # (50000, 3, 32, 32)
y_all = torch.tensor(cifar_train.targets)

perm = torch.randperm(len(X_all), generator=torch.Generator().manual_seed(SEED))
train_idx, val_idx = perm[:45000], perm[45000:]

MEAN = X_all[train_idx].mean(dim=(0, 2, 3))     # statistics of the 45k training split only
STD = X_all[train_idx].std(dim=(0, 2, 3))

def normalize(x):
    return (x - MEAN.view(1, 3, 1, 1).to(x.device)) / STD.view(1, 3, 1, 1).to(x.device)

def denormalize(x):
    return x * STD.view(1, 3, 1, 1).to(x.device) + MEAN.view(1, 3, 1, 1).to(x.device)

X_train, y_train = normalize(X_all[train_idx]).to(device), y_all[train_idx].to(device)
X_val, y_val = normalize(X_all[val_idx]).to(device), y_all[val_idx].to(device)
del X_all

if QUICK:   # smoke-test mode: small subsets so that every cell finishes quickly
    X_train, y_train = X_train[:5000], y_train[:5000]
    X_val, y_val = X_val[:1000], y_val[:1000]

print('train', tuple(X_train.shape), '| validation', tuple(X_val.shape))
print('channel mean (train split):', MEAN.numpy().round(4), '| channel std:', STD.numpy().round(4))
print('after normalisation -> train mean', X_train.mean((0, 2, 3)).cpu().numpy().round(3),
      '| train std', X_train.std((0, 2, 3)).cpu().numpy().round(3))
print('validation images per class:', torch.bincount(y_val, minlength=10).tolist())

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(13, 3))
for c in range(10):
    idx = (y_train == c).nonzero()[:2, 0]
    for r in range(2):
        img = denormalize(X_train[idx[r]][None])[0].permute(1, 2, 0).clamp(0, 1).cpu()
        axes[r, c].imshow(img)
        axes[r, c].axis('off')
    axes[0, c].set_title(CLASSES[c], fontsize=9)
plt.suptitle('CIFAR-10 training images (two per class)')
plt.tight_layout()
plt.show()

### Shared utilities
* `iterate_minibatches`: shuffles and slices tensors that are already on the GPU.
* Augmentations: `random_flip`, `random_crop` (4-pixel zero padding), `cutout` and `mixup`. All of them are implemented by hand and act on a whole GPU batch at once.
* `train_model`: one training loop used by Tasks 2–4. It supports a per-iteration learning-rate schedule, augmentation, mixup, label smoothing and fp16 mixed precision. It records the training loss, the validation loss and accuracy, and (optionally) the accuracy on the clean training set.

In [ ]:
def iterate_minibatches(X, y, batch_size, shuffle=True, drop_last=False):
    n = len(X)
    order = torch.randperm(n, device=X.device) if shuffle else torch.arange(n, device=X.device)
    stop = n - n % batch_size if drop_last else n
    for i in range(0, stop, batch_size):
        idx = order[i:i + batch_size]
        yield X[idx], y[idx]

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def autocast(enabled):
    """fp16 autocast on the GPU, or a no-op context."""
    if enabled and device.type == 'cuda':
        return torch.autocast('cuda', dtype=torch.float16)
    return contextlib.nullcontext()

def make_grad_scaler(enabled):
    if hasattr(torch.amp, 'GradScaler'):
        return torch.amp.GradScaler('cuda', enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)

# ---------------- data augmentation on GPU batches ----------------
def random_flip(x):
    """Horizontal flip of each image with probability 1/2."""
    flip = torch.rand(x.shape[0], device=x.device) < 0.5
    return torch.where(flip[:, None, None, None], x.flip(3), x)

def random_crop(x, pad=4):
    """Zero-pad by `pad` pixels and take an independent random 32x32 crop of each image."""
    B, C, H, W = x.shape
    xp = F.pad(x, (pad, pad, pad, pad))           # 0 = channel mean after normalisation
    dy = torch.randint(0, 2 * pad + 1, (B,), device=x.device)
    dx = torch.randint(0, 2 * pad + 1, (B,), device=x.device)
    rows = (dy[:, None] + torch.arange(H, device=x.device))[:, None, :, None]   # (B,1,H,1)
    cols = (dx[:, None] + torch.arange(W, device=x.device))[:, None, None, :]   # (B,1,1,W)
    b = torch.arange(B, device=x.device)[:, None, None, None]
    c = torch.arange(C, device=x.device)[None, :, None, None]
    return xp[b, c, rows, cols]                    # (B,C,H,W) via advanced indexing

def cutout(x, size=16):
    """Cutout (DeVries & Taylor, 2017): set a random size x size square of every image to 0.
    The centre is uniform over the image, so the square may be partly outside the image."""
    B, C, H, W = x.shape
    cy = torch.randint(0, H, (B, 1), device=x.device)
    cx = torch.randint(0, W, (B, 1), device=x.device)
    ys = torch.arange(H, device=x.device)[None, :]
    xs = torch.arange(W, device=x.device)[None, :]
    in_y = (ys >= cy - size // 2) & (ys < cy + size // 2)          # (B,H)
    in_x = (xs >= cx - size // 2) & (xs < cx + size // 2)          # (B,W)
    mask = in_y[:, :, None] & in_x[:, None, :]                     # (B,H,W)
    return x.masked_fill(mask[:, None], 0.0)

def mixup(x, y, alpha):
    """Mixup (Zhang et al., 2018): x~ = lam*x_i + (1-lam)*x_j, lam ~ Beta(alpha, alpha).
    Returns the mixed batch, both label sets and lam (the loss is mixed with the same lam)."""
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(x.shape[0], device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

AUGMENTATIONS = {'flip': random_flip, 'crop': random_crop,
                 'cutout': cutout, 'cutout8': lambda x: cutout(x, 8)}

def make_augment(names):
    """Compose a list of augmentation names into one function (None for no augmentation)."""
    if not names:
        return None
    def augment(x):
        for name in names:
            x = AUGMENTATIONS[name](x)
        return x
    return augment

def cosine_schedule(base_lr, warmup_frac=0.0):
    """Linear warm-up for warmup_frac of the steps, then cosine decay to 0 (per iteration)."""
    def lr_at(step, total):
        warm = int(warmup_frac * total)
        if step < warm:
            return base_lr * (step + 1) / warm
        t = (step - warm) / max(1, total - warm)
        return 0.5 * base_lr * (1 + math.cos(math.pi * t))
    return lr_at

# ---------------- evaluation and the generic training loop ----------------
@torch.no_grad()
def evaluate(model, X, y, amp=USE_AMP, batch_size=1000):
    """Mean cross-entropy and accuracy of `model` on (X, y) in eval mode (no augmentation)."""
    model.eval()
    total_loss, correct = 0.0, 0
    for i in range(0, len(X), batch_size):
        xb, yb = X[i:i + batch_size], y[i:i + batch_size]
        with autocast(amp):
            out = model(xb)
        total_loss += F.cross_entropy(out.float(), yb, reduction='sum').item()
        correct += (out.argmax(1) == yb).sum().item()
    return total_loss / len(X), correct / len(X)

def train_model(model, optimizer, X, y, epochs, batch_size=128, lr_schedule=None,
                augment=None, mixup_alpha=0.0, label_smoothing=0.0, amp=False,
                X_val=None, y_val=None, eval_train=False, log_every=1):
    """Train `model` and return a history dict (lists indexed by epoch).
    lr_schedule(step, total_steps) -> lr is applied every iteration (None = constant lr).
    eval_train=True also measures accuracy on the clean, un-augmented training set."""
    amp = amp and device.type == 'cuda'
    scaler = make_grad_scaler(amp)
    total_steps = epochs * (len(X) // batch_size)
    hist = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'train_eval_acc': [], 'time': [],
            'diverged': False}
    step, t0 = 0, time.time()
    for epoch in range(epochs):
        model.train()
        running, n_seen = torch.zeros((), device=X.device), 0
        for xb, yb in iterate_minibatches(X, y, batch_size, shuffle=True, drop_last=True):
            if lr_schedule is not None:
                for group in optimizer.param_groups:
                    group['lr'] = lr_schedule(step, total_steps)
            if augment is not None:
                xb = augment(xb)
            with autocast(amp):
                if mixup_alpha > 0:
                    xb, ya, yb2, lam = mixup(xb, yb, mixup_alpha)
                    out = model(xb)
                    loss = (lam * F.cross_entropy(out, ya, label_smoothing=label_smoothing)
                            + (1 - lam) * F.cross_entropy(out, yb2, label_smoothing=label_smoothing))
                else:
                    out = model(xb)
                    loss = F.cross_entropy(out, yb, label_smoothing=label_smoothing)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += loss.detach().float() * len(xb)
            n_seen += len(xb)
            step += 1
        train_loss = running.item() / max(n_seen, 1)
        hist['train_loss'].append(train_loss)
        if not math.isfinite(train_loss):
            hist['diverged'] = True
        msg = f'  epoch {epoch + 1:3d}/{epochs}  train loss {train_loss:.4f}'
        if X_val is not None:
            val_loss, val_acc = evaluate(model, X_val, y_val, amp=amp)
            hist['val_loss'].append(val_loss)
            hist['val_acc'].append(val_acc)
            msg += f'  val acc {val_acc:.4f}'
        if eval_train:
            hist['train_eval_acc'].append(evaluate(model, X, y, amp=amp)[1])
            msg += f'  train acc {hist["train_eval_acc"][-1]:.4f}'
        hist['time'].append(time.time() - t0)
        if (epoch + 1) % log_every == 0 or epoch == epochs - 1 or hist['diverged']:
            print(msg + f'  [{hist["time"][-1]:.0f} s]')
        if hist['diverged']:
            print('  -> the loss is not finite: training diverged, run stopped.')
            break
    return hist

In [ ]:
# A look at the augmentations used in Task 3 (top: original images).
torch.manual_seed(1); np.random.seed(1)
demo = X_train[:8]
mixed, _, _, lam = mixup(demo, y_train[:8], alpha=0.4)
rows = {'original': demo, 'flip + crop': random_crop(random_flip(demo)),
        'cutout (16x16)': cutout(demo), f'mixup (lam={lam:.2f})': mixed}
fig, axes = plt.subplots(len(rows), 8, figsize=(11, 6))
for r, (name, imgs) in enumerate(rows.items()):
    for c in range(8):
        axes[r, c].imshow(denormalize(imgs[c:c + 1])[0].permute(1, 2, 0).clamp(0, 1).cpu())
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([]); axes[r, c].grid(False)
    axes[r, 0].set_ylabel(name, fontsize=9)
plt.suptitle('Data augmentation implemented on GPU tensors')
plt.tight_layout()
plt.show()

---
## Task 1 — Backpropagation from scratch

### 1(a) Layers with explicit forward and backward passes
Every layer stores what it needs during `forward` and returns $\partial L/\partial(\text{input})$ in `backward`. Here $G$ is the upstream gradient.

| Layer | Forward | Backward |
|---|---|---|
| Linear | $Y = XW^\top + b$ | $\dfrac{\partial L}{\partial W} = G^\top X,\quad \dfrac{\partial L}{\partial b} = \sum_n G_n,\quad \dfrac{\partial L}{\partial X} = G\,W$ |
| ReLU | $y = \max(0, x)$ | $\dfrac{\partial L}{\partial x} = G \odot \mathbb{1}[x>0]$ |
| Sigmoid | $y = \sigma(x) = 1/(1+e^{-x})$ | $\dfrac{\partial L}{\partial x} = G \odot y(1-y)$ |
| Softmax + cross-entropy | $L = -\frac1N\sum_n \log p_{n,y_n}$, $p = \mathrm{softmax}(z)$ | $\dfrac{\partial L}{\partial z} = \frac1N\,(p - \mathrm{onehot}(y))$ |

The softmax subtracts the row maximum before exponentiating, so it cannot overflow. No `requires_grad` tensors are used, and the training loop runs inside `torch.no_grad()`.

In [ ]:
class Linear:
    """Fully connected layer  y = x W^T + b  (W has shape (out, in), as in nn.Linear)."""
    def __init__(self, n_in, n_out, init='he', dtype=torch.float32, device='cpu'):
        std = {'he': math.sqrt(2.0 / n_in),                    # Kaiming/He, for ReLU
               'xavier': math.sqrt(2.0 / (n_in + n_out))}[init] # Glorot/Xavier, for sigmoid/tanh
        self.W = torch.randn(n_out, n_in, dtype=dtype, device=device) * std
        self.b = torch.zeros(n_out, dtype=dtype, device=device)
        self.dW, self.db = torch.zeros_like(self.W), torch.zeros_like(self.b)

    def forward(self, x):
        self.x = x                           # cache the input for backward
        return x @ self.W.T + self.b

    def backward(self, grad_out):            # grad_out = dL/dy, shape (N, out)
        self.dW = grad_out.T @ self.x        # (out, in)
        self.db = grad_out.sum(dim=0)        # (out,)
        return grad_out @ self.W             # dL/dx, shape (N, in)

    def params(self): return [self.W, self.b]
    def grads(self): return [self.dW, self.db]


class ReLU:
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask
    def backward(self, grad_out):
        return grad_out * self.mask          # derivative is 1 where x > 0 and 0 elsewhere
    def params(self): return []
    def grads(self): return []


class Sigmoid:
    def forward(self, x):
        self.out = 1.0 / (1.0 + torch.exp(-x))
        return self.out
    def backward(self, grad_out):
        return grad_out * self.out * (1.0 - self.out)    # sigma' = sigma (1 - sigma) <= 1/4
    def params(self): return []
    def grads(self): return []


class SoftmaxCrossEntropy:
    """Mean cross-entropy between softmax(logits) and integer labels."""
    def forward(self, logits, y):
        z = logits - logits.max(dim=1, keepdim=True).values             # numerical stability
        log_p = z - torch.log(torch.exp(z).sum(dim=1, keepdim=True))     # log-softmax
        self.p, self.y = torch.exp(log_p), y
        return -log_p[torch.arange(len(y), device=y.device), y].mean()
    def backward(self):
        grad = self.p.clone()
        grad[torch.arange(len(self.y), device=self.y.device), self.y] -= 1.0   # p - onehot(y)
        return grad / len(self.y)                                          # mean over the batch


class MLP:
    """Linear -> activation -> ... -> Linear, trained with our own backward pass."""
    def __init__(self, sizes, act='relu', init='he', dtype=torch.float32, device='cpu'):
        self.layers = []
        for i, (n_in, n_out) in enumerate(zip(sizes[:-1], sizes[1:])):
            self.layers.append(Linear(n_in, n_out, init, dtype, device))
            if i < len(sizes) - 2:                        # no activation after the output layer
                self.layers.append(ReLU() if act == 'relu' else Sigmoid())
        self.criterion = SoftmaxCrossEntropy()

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def loss(self, x, y):
        return self.criterion.forward(self.forward(x), y)

    def backward(self):
        grad = self.criterion.backward()
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad                                       # dL/d(input)

    def params(self): return [p for layer in self.layers for p in layer.params()]
    def grads(self): return [g for layer in self.layers for g in layer.grads()]
    def linear_layers(self): return [l for l in self.layers if isinstance(l, Linear)]

    def sgd_step(self, lr):
        for p, g in zip(self.params(), self.grads()):
            p -= lr * g

### 1(b) Gradient check against autograd and centred finite differences
A small MLP (20 → 32 → 16 → 10, batch of 8) is built in **float64** with ReLU and with sigmoid. For every parameter tensor, and for the input, we compare our gradient with:
* **autograd**: the same network written with differentiable `torch` operations, followed by `loss.backward()`;
* **centred finite differences**: $\dfrac{\partial L}{\partial \theta_i} \approx \dfrac{L(\theta + h e_i) - L(\theta - h e_i)}{2h}$ with $h = 10^{-5}$. Its truncation error is $O(h^2)$.

The relative error is $\dfrac{\lVert g_{\text{ours}} - g_{\text{ref}}\rVert_2}{\lVert g_{\text{ours}}\rVert_2 + \lVert g_{\text{ref}}\rVert_2}$, and `torch.allclose` gives the pass/fail decision. Float64 is required for the finite differences. In float32, the round-off error of the difference quotient, about $\epsilon_{32} L / h \approx 10^{-2}$, would be as large as the gradients themselves.

In [ ]:
def rel_error(a, b):
    """||a - b|| / (||a|| + ||b||); 0 means identical."""
    return ((a - b).norm() / (a.norm() + b.norm()).clamp_min(1e-30)).item()

def autograd_reference(net, x, y, act):
    """The same network written with differentiable torch ops; returns autograd's gradients."""
    params = [p.clone().requires_grad_(True) for p in net.params()]
    x = x.clone().requires_grad_(True)
    h = x
    for i in range(0, len(params), 2):
        h = h @ params[i].T + params[i + 1]
        if i < len(params) - 2:
            h = torch.relu(h) if act == 'relu' else torch.sigmoid(h)
    loss = F.cross_entropy(h, y)
    loss.backward()
    return loss.item(), [p.grad for p in params] + [x.grad]

def finite_difference(net, x, y, tensor, h=1e-5):
    """Centred differences for every element of `tensor` (perturbed in place, then restored)."""
    grad = torch.zeros_like(tensor)
    flat, gflat = tensor.view(-1), grad.view(-1)
    for i in range(flat.numel()):
        old = flat[i].item()
        flat[i] = old + h; loss_plus = net.loss(x, y).item()
        flat[i] = old - h; loss_minus = net.loss(x, y).item()
        flat[i] = old
        gflat[i] = (loss_plus - loss_minus) / (2 * h)
    return grad

rows = []
for act in ['relu', 'sigmoid']:
    torch.manual_seed(0)
    net = MLP([20, 32, 16, 10], act=act, dtype=torch.float64)
    x = torch.randn(8, 20, dtype=torch.float64)
    y = torch.randint(0, 10, (8,))
    our_loss = net.loss(x, y).item()
    dx = net.backward()
    ours = net.grads() + [dx]
    ag_loss, reference = autograd_reference(net, x, y, act)
    names = ['W1', 'b1', 'W2', 'b2', 'W3', 'b3', 'x (input)']
    for name, g, g_ag, t in zip(names, ours, reference, net.params() + [x]):
        g_fd = finite_difference(net, x, y, t)
        rows.append({'activation': act, 'tensor': name, 'shape': str(tuple(t.shape)),
                     'rel. error vs autograd': f'{rel_error(g, g_ag):.2e}',
                     'allclose (autograd)': torch.allclose(g, g_ag),
                     'rel. error vs finite diff.': f'{rel_error(g, g_fd):.2e}',
                     'allclose (finite diff.)': torch.allclose(g, g_fd, rtol=1e-5, atol=1e-8)})
    print(f'{act:8s}: our loss = {our_loss:.12f} | autograd loss = {ag_loss:.12f}')

grad_check = pd.DataFrame(rows)
display(grad_check)
print('All gradients match autograd          :', bool(grad_check['allclose (autograd)'].all()))
print('All gradients match finite differences:', bool(grad_check['allclose (finite diff.)'].all()))

**Interpretation.** For both activations, every parameter gradient and the input gradient agree with autograd to ‹~1e-16› relative error, which is float64 round-off. Our backward pass therefore performs the same computation as autograd. The agreement with centred finite differences is ‹~1e-10›. That is the expected accuracy of a numerical derivative: the $O(h^2)$ truncation error plus a round-off term of order $\epsilon_{64}\,L/h$. `torch.allclose` returns `True` everywhere. The input gradient $\partial L/\partial x$ also matches, which shows that the layers chain correctly: each layer passes the right gradient to the layer below it.

### 1(c) Training the from-scratch MLP on flattened CIFAR-10
Architecture: 3072 → 512 → 256 → 10 with ReLU and He initialisation. Training uses plain mini-batch SGD (batch 128, initial learning rate 0.05 with cosine decay) for 25 epochs. Only our own `backward()` and `sgd_step()` are used.

In [ ]:
set_seed()
Xf_train, Xf_val = X_train.flatten(1), X_val.flatten(1)     # (N, 3072)
mlp = MLP([3072, 512, 256, 10], act='relu', init='he', device=device)
EPOCHS_MLP, BS_MLP, LR_MLP = (2 if QUICK else 25), 128, 0.05

def mlp_evaluate(net, X, y, batch_size=5000):
    total, correct = 0.0, 0
    for i in range(0, len(X), batch_size):
        logits = net.forward(X[i:i + batch_size])
        total += net.criterion.forward(logits, y[i:i + batch_size]).item() * len(logits)
        correct += (logits.argmax(1) == y[i:i + batch_size]).sum().item()
    return total / len(X), correct / len(X)

steps_per_epoch = len(Xf_train) // BS_MLP
total_steps, step = EPOCHS_MLP * steps_per_epoch, 0
iter_loss = []
hist_mlp = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
t0 = time.time()
with torch.no_grad():                    # autograd is off: only our own backward pass is used
    for epoch in range(EPOCHS_MLP):
        for xb, yb in iterate_minibatches(Xf_train, y_train, BS_MLP, drop_last=True):
            lr = 0.5 * LR_MLP * (1 + math.cos(math.pi * step / total_steps))   # cosine decay
            loss = mlp.loss(xb, yb)
            mlp.backward()
            mlp.sgd_step(lr)
            iter_loss.append(loss.item())
            step += 1
        tr_loss, tr_acc = mlp_evaluate(mlp, Xf_train, y_train)
        va_loss, va_acc = mlp_evaluate(mlp, Xf_val, y_val)
        for k, v in zip(hist_mlp, (tr_loss, tr_acc, va_loss, va_acc)):
            hist_mlp[k].append(v)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'epoch {epoch + 1:2d}: train loss {tr_loss:.4f}  train acc {tr_acc:.4f}  '
                  f'val loss {va_loss:.4f}  val acc {va_acc:.4f}  [{time.time() - t0:.0f} s]')

print(f'\nFinal validation accuracy of the from-scratch MLP: {100 * hist_mlp["val_acc"][-1]:.2f} %'
      f'  (best epoch: {100 * max(hist_mlp["val_acc"]):.2f} %)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
it = np.arange(1, len(iter_loss) + 1) / steps_per_epoch
ep = np.arange(1, EPOCHS_MLP + 1)
axes[0].plot(it, iter_loss, color=PALETTE[0], alpha=0.25, lw=0.6, label='mini-batch loss')
axes[0].plot(it, pd.Series(iter_loss).rolling(100, min_periods=1).mean(), color=PALETTE[0],
             label='mini-batch loss (moving average, 100 it.)')
axes[0].plot(ep, hist_mlp['val_loss'], 'o-', color=PALETTE[1], ms=4, label='validation loss')
axes[0].set(xlabel='epoch', ylabel='cross-entropy', title='From-scratch MLP: loss')
axes[0].legend()
axes[1].plot(ep, 100 * np.array(hist_mlp['train_acc']), 'o-', ms=4, label='train')
axes[1].plot(ep, 100 * np.array(hist_mlp['val_acc']), 'o-', ms=4, label='validation')
axes[1].set(xlabel='epoch', ylabel='accuracy (%)', title='From-scratch MLP: accuracy')
axes[1].legend()
plt.tight_layout()
plt.show()

**Interpretation.** The mini-batch loss drops quickly in the first epochs and then decreases more slowly; the cosine decay of the learning rate smooths the end of training. The final validation accuracy is ‹xx.x %›, far above the 10 % chance level, but the training accuracy keeps rising (‹xx %›) while the validation loss levels off or rises after about epoch ‹k›. The MLP is overfitting. This is expected: a fully connected network on flattened pixels has no notion of locality or translation invariance. It must learn each pattern separately at every image position with about 1.7 M parameters. CNNs build these priors in through weight sharing and local receptive fields, which is why they do much better on the same data (Tasks 2–4).

### 1(d) Per-layer gradient norms at initialisation in a 10-layer MLP
The network is 3072 → 256 (×9) → 10, i.e. 10 `Linear` layers. We do one forward and one backward pass on 512 training images and record $\lVert \partial L/\partial W_\ell \rVert_F$ for every layer. The results are averaged over 5 random initialisations; the shaded band shows the minimum and maximum. Each activation is shown with its recommended initialisation (sigmoid with Xavier, ReLU with He) and with the other one, so that the effect of the activation can be separated from the effect of the initialisation.

In [ ]:
def layer_grad_norms(act, init, seed, n=512):
    torch.manual_seed(seed)
    net = MLP([3072] + [256] * 9 + [10], act=act, init=init, device=device)   # 10 Linear layers
    with torch.no_grad():
        net.loss(X_train[:n].flatten(1), y_train[:n])
        net.backward()
    return [layer.dW.norm().item() for layer in net.linear_layers()]

settings = [('relu', 'he'), ('relu', 'xavier'), ('sigmoid', 'xavier'), ('sigmoid', 'he')]
grad_norms = {s: np.array([layer_grad_norms(*s, seed) for seed in range(5)]) for s in settings}

fig, ax = plt.subplots(figsize=(8.5, 4.5))
layers = np.arange(1, 11)
for (act, init), marker in zip(settings, ['o', 's', '^', 'D']):
    g = grad_norms[(act, init)]
    line, = ax.semilogy(layers, g.mean(0), marker=marker, ms=6, label=f'{act} + {init} init')
    ax.fill_between(layers, g.min(0), g.max(0), color=line.get_color(), alpha=0.15)
ax.set(xlabel='layer (1 = closest to the input)', ylabel=r'$\|\partial L/\partial W_\ell\|_F$ (log scale)',
       title='Per-layer gradient norm at initialisation, 10-layer MLP', xticks=layers)
ax.legend()
plt.tight_layout()
plt.show()

table = pd.DataFrame({f'{a}+{i}': grad_norms[(a, i)].mean(0) for a, i in settings}, index=layers)
table.index.name = 'layer'
display(table.map(lambda v: f'{v:.2e}') if hasattr(table, 'map') else table.applymap(lambda v: f'{v:.2e}'))
for s in settings:
    g = grad_norms[s].mean(0)
    print(f'{s[0]:7s}+{s[1]:6s}: ||dW_1|| / ||dW_10|| = {g[0] / g[-1]:.2e}')

**Interpretation.** Backpropagation through one hidden layer multiplies the error signal by $W_\ell^\top$ and then by the activation derivative: $\delta_{\ell-1} = \big(W_\ell^\top \delta_\ell\big)\odot f'(z_{\ell-1})$.
* **Sigmoid:** $\sigma'(z) = \sigma(z)(1-\sigma(z)) \le 1/4$, and it is about 0.2 for typical pre-activations. With Xavier initialisation, $W^\top$ roughly preserves the variance of the signal, so every layer shrinks the gradient by a factor of about 4–5. After 9 hidden layers, the first layer's gradient is ‹~10⁻⁵–10⁻⁶› times that of the last layer: the **vanishing gradient** problem. The early layers would barely learn. He initialisation (larger weights) slows the decay, but cannot remove it because the derivative of the sigmoid is always below 1/4. The sigmoid also outputs values with mean 0.5 rather than 0, which skews the updates further.
* **ReLU + He:** $f'(z) \in \{0, 1\}$ and about half of the units are active. He initialisation, $\mathrm{Var}(W) = 2/n_{\text{in}}$, compensates exactly for that factor 1/2, so the variance of the backward signal stays roughly constant and the gradient norms are of the same order in all 10 layers. With Xavier initialisation, ReLU loses about a factor $\sqrt{2}$ per layer, which gives a milder but visible decay. Both the activation and the initialisation therefore matter, and ReLU with He initialisation is the combination that makes deep networks trainable from the first step.

---
## Task 2 — Optimisers

### 2(a) SGD with momentum and Adam as `torch.optim.Optimizer` subclasses
**SGD with momentum** (heavy ball, as implemented in PyTorch):
$g_t = \nabla L(\theta_t) + \lambda\theta_t,\quad v_t = \mu v_{t-1} + g_t \;(v_1 = g_1),\quad \theta_{t+1} = \theta_t - \eta\, v_t.$
With $\mu = 0$, this is plain SGD.

**Adam** with bias correction:
$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t,\quad v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2,\quad \hat m_t = \dfrac{m_t}{1-\beta_1^t},\quad \hat v_t = \dfrac{v_t}{1-\beta_2^t},\quad \theta_{t+1} = \theta_t - \eta\,\dfrac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}.$
Because $m_0 = v_0 = 0$, the raw moments are biased towards zero in the first steps. Dividing by $1-\beta^t$ removes that bias.

In [ ]:
class MySGD(torch.optim.Optimizer):
    """SGD with heavy-ball momentum, the same update as torch.optim.SGD (dampening=0, nesterov=False).
    momentum=0 gives plain SGD."""
    def __init__(self, params, lr=0.01, momentum=0.0, weight_decay=0.0):
        if lr < 0 or momentum < 0 or weight_decay < 0:
            raise ValueError('hyper-parameters must be non-negative')
        super().__init__(params, dict(lr=lr, momentum=momentum, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, mu, wd = group['lr'], group['momentum'], group['weight_decay']
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                if wd != 0:
                    g = g.add(p, alpha=wd)                 # gradient of (wd/2)||p||^2 added
                if mu != 0:
                    state = self.state[p]
                    if 'velocity' not in state:
                        state['velocity'] = g.clone()      # v_1 = g_1 (as in PyTorch)
                    else:
                        state['velocity'].mul_(mu).add_(g) # v_t = mu v_{t-1} + g_t
                    g = state['velocity']
                p.add_(g, alpha=-lr)                       # theta <- theta - lr * v
        return loss


class MyAdam(torch.optim.Optimizer):
    """Adam (Kingma & Ba, 2015) with bias correction, the same update as torch.optim.Adam."""
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        if lr < 0 or eps < 0 or not (0 <= betas[0] < 1 and 0 <= betas[1] < 1):
            raise ValueError('invalid hyper-parameters')
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, (beta1, beta2), eps, wd = group['lr'], group['betas'], group['eps'], group['weight_decay']
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                if wd != 0:
                    g = g.add(p, alpha=wd)
                state = self.state[p]
                if len(state) == 0:
                    state['t'] = 0
                    state['m'] = torch.zeros_like(p)       # first moment  (mean of g)
                    state['v'] = torch.zeros_like(p)       # second moment (mean of g^2)
                state['t'] += 1
                t, m, v = state['t'], state['m'], state['v']
                m.mul_(beta1).add_(g, alpha=1 - beta1)
                v.mul_(beta2).addcmul_(g, g, value=1 - beta2)
                m_hat = m / (1 - beta1 ** t)               # bias correction
                v_hat = v / (1 - beta2 ** t)
                p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)
        return loss

**Do they match PyTorch?** Two copies of a small CNN with identical initial weights are trained side by side for 100 steps on the same mini-batches, one with our optimiser and one with the `torch.optim` version. After every step we record the largest absolute difference between the two sets of parameters. The test runs in float64 so that only the algorithms, and not round-off, are compared.

In [ ]:
def tiny_cnn():
    return nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(4),
                         nn.Flatten(), nn.Linear(8 * 8 * 8, 10))

def compare_with_torch(make_ours, make_ref, steps=100):
    torch.manual_seed(0)
    model_a = tiny_cnn().double()
    model_b = copy.deepcopy(model_a)                              # identical initial weights
    opt_a, opt_b = make_ours(model_a.parameters()), make_ref(model_b.parameters())
    X, y = X_train[:512].double().cpu(), y_train[:512].cpu()
    losses_a, losses_b, max_diff = [], [], []
    for s in range(steps):
        idx = torch.arange(s * 64, (s + 1) * 64) % len(X)          # same mini-batch for both
        for model, opt, losses in ((model_a, opt_a, losses_a), (model_b, opt_b, losses_b)):
            opt.zero_grad()
            loss = F.cross_entropy(model(X[idx]), y[idx])
            loss.backward()
            opt.step()
            losses.append(loss.item())
        max_diff.append(max((pa - pb).abs().max().item()
                            for pa, pb in zip(model_a.parameters(), model_b.parameters())))
    match = all(torch.allclose(pa, pb) for pa, pb in zip(model_a.parameters(), model_b.parameters()))
    return losses_a, losses_b, max_diff, match

OPT_TESTS = {
    'SGD (lr=0.05)': (lambda p: MySGD(p, lr=0.05),
                      lambda p: torch.optim.SGD(p, lr=0.05)),
    'SGD + momentum (lr=0.02, mu=0.9)': (lambda p: MySGD(p, lr=0.02, momentum=0.9),
                                         lambda p: torch.optim.SGD(p, lr=0.02, momentum=0.9)),
    'SGD + momentum + wd (5e-4)': (lambda p: MySGD(p, lr=0.02, momentum=0.9, weight_decay=5e-4),
                                   lambda p: torch.optim.SGD(p, lr=0.02, momentum=0.9, weight_decay=5e-4)),
    'Adam (lr=1e-3)': (lambda p: MyAdam(p, lr=1e-3),
                       lambda p: torch.optim.Adam(p, lr=1e-3)),
    'Adam + wd (1e-4)': (lambda p: MyAdam(p, lr=1e-3, weight_decay=1e-4),
                         lambda p: torch.optim.Adam(p, lr=1e-3, weight_decay=1e-4)),
}
rows, curves = [], {}
for name, (mine, ref) in OPT_TESTS.items():
    la, lb, diff, match = compare_with_torch(mine, ref)
    curves[name] = (la, lb)
    rows.append({'optimiser': name, 'steps': len(la), 'final loss (ours)': f'{la[-1]:.6f}',
                 'final loss (torch)': f'{lb[-1]:.6f}', 'max |θ_ours − θ_torch| (any step)': f'{max(diff):.1e}',
                 'torch.allclose': match})
display(pd.DataFrame(rows))

fig, ax = plt.subplots(figsize=(8.5, 4))
for (name, (la, lb)), color in zip(curves.items(), PALETTE):
    ax.plot(la, color=color, label=f'{name}')
    ax.plot(range(0, len(lb), 5), lb[::5], 'o', color=color, ms=4, mfc='none')
ax.set(xlabel='step', ylabel='training loss', title='Ours (lines) vs torch.optim (circles)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Interpretation.** For all five settings, the parameters of the two copies stay identical to ‹~1e-16›, i.e. float64 round-off, and `torch.allclose` is `True`; the loss curves lie on top of each other. Our classes therefore implement exactly the PyTorch update rules, including the details: the momentum buffer is initialised with the first gradient, weight decay is added to the gradient as an L2 term, and Adam's bias correction is applied before $\epsilon$ is added. Because they subclass `torch.optim.Optimizer`, they also work with parameter groups, learning-rate changes and `GradScaler`. They are used in all experiments below.

### 2(b) Trajectories on $f(x, y) = \tfrac12 (x^2 + 50y^2)$
The Hessian is $\mathrm{diag}(1, 50)$, so the condition number is $\kappa = 50$. Gradient descent with step $\eta$ multiplies each coordinate by $(1 - \eta\lambda_i)$ at every step, and it is stable only when $\eta < 2/\lambda_{\max} = 0.04$. All methods start at $(-10, 1)$ and run for 150 iterations. Here "SGD" is full-gradient descent, because $f$ has no noise.

In [ ]:
def f_quad(p):
    return 0.5 * (p[0] ** 2 + 50 * p[1] ** 2)

def trajectory(make_opt, start=(-10.0, 1.0), steps=150):
    p = torch.tensor(start, dtype=torch.float64, requires_grad=True)
    opt = make_opt([p])
    path = [p.detach().clone().numpy()]
    for _ in range(steps):
        opt.zero_grad()
        f_quad(p).backward()                 # gradient (x, 50 y)
        opt.step()
        path.append(p.detach().clone().numpy())
    return np.array(path)

runs_2b = {
    'SGD (lr=0.035)': lambda ps: MySGD(ps, lr=0.035),
    'SGD + momentum (lr=0.035, mu=0.9)': lambda ps: MySGD(ps, lr=0.035, momentum=0.9),
    'Adam (lr=0.3)': lambda ps: MyAdam(ps, lr=0.3),
}
paths = {name: trajectory(make) for name, make in runs_2b.items()}
diverging = trajectory(lambda ps: MySGD(ps, lr=0.041))   # just above 2 / lambda_max

xs, ys = np.linspace(-11, 11, 400), np.linspace(-1.6, 1.6, 400)
XX, YY = np.meshgrid(xs, ys)
ZZ = 0.5 * (XX ** 2 + 50 * YY ** 2)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={'width_ratios': [1.7, 1]})
ax = axes[0]
cs = ax.contour(XX, YY, ZZ, levels=np.logspace(-2, 2.2, 14), colors='#898781', linewidths=0.7)
ax.clabel(cs, fmt='%.2g', fontsize=7)
for (name, path), marker in zip(paths.items(), ['o', 's', '^']):
    ax.plot(path[:, 0], path[:, 1], marker=marker, ms=3, lw=1.2, label=name)
ax.plot(-10, 1, 'ks', ms=8, label='start (-10, 1)')
ax.plot(0, 0, 'k*', ms=14, label='minimum')
ax.set(xlim=(-11, 11), ylim=(-1.6, 1.6), xlabel='x', ylabel='y',
       title=r'Trajectories on $f(x,y)=\frac{1}{2}(x^2+50y^2)$')
ax.legend(loc='lower right', fontsize=8)
ax.grid(False)

ax = axes[1]
for name, path in paths.items():
    ax.semilogy(f_quad(path.T), label=name)
ax.semilogy(f_quad(diverging.T), '--', color=PALETTE[7], label='SGD (lr=0.041 > 2/50): diverges')
ax.set(xlabel='iteration', ylabel='f (log scale)', title='Objective value', ylim=(1e-12, 1e4))
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

summary = []
for name, path in paths.items():
    f = f_quad(path.T)
    hit = np.nonzero(f < 1e-3)[0]
    summary.append({'method': name, 'iterations to f < 1e-3': int(hit[0]) if len(hit) else '> 150',
                    'f after 150 iterations': f'{f[-1]:.2e}'})
display(pd.DataFrame(summary))

**Interpretation.**
* **SGD:** the per-step factors are $1-\eta\lambda$: $1 - 0.035\cdot 50 = -0.75$ along $y$ and $1 - 0.035 = 0.965$ along $x$. The steep $y$ direction flips sign at every step, which produces the **zig-zag** across the narrow valley, while the shallow $x$ direction crawls. The learning rate is capped by the steepest direction ($\eta < 2/50$; the dashed run at $\eta = 0.041$ diverges), and that cap makes progress along the flat direction slow. This is the effect of ill-conditioning.
* **SGD with momentum:** the velocity accumulates the consistent $x$-component of the gradient and averages out the alternating $y$-component. Both directions now contract at about $\sqrt{\mu} \approx 0.95$ per step instead of 0.965 along $x$, so the method reaches ‹f < 10⁻³ in N iterations› versus ‹M› for SGD. The price is **overshoot**: the heavy ball passes the minimum in $x$ and spirals back in.
* **Adam:** dividing by $\sqrt{\hat v}$ normalises each coordinate, so the first steps are about $\eta\,\mathrm{sign}(g)$ and the trajectory heads **diagonally**, ignoring the 50× difference in curvature. Its progress along $x$ does not depend on the small curvature. Near the minimum, the step size is set by $\eta$ rather than by the gradient, so Adam oscillates around the optimum until the momentum and the slowly decaying $\hat v$ damp the motion.

### 2(c) Small CNN with each optimiser at three learning rates
`SmallCNN`: three blocks of conv3×3 → ReLU → max-pool (32/64/128 channels), followed by FC 2048 → 256 → 10. It has no batch normalisation here, so the optimisers are compared without that help. Each optimiser is trained for 20 epochs with batch size 128, a **constant** learning rate and no augmentation, at the same learning rates $\{10^{-3}, 10^{-2}, 10^{-1}\}$. A common logarithmic grid makes the sensitivity of the three methods directly comparable. Every run starts from the same initial weights.

In [ ]:
class SmallCNN(nn.Module):
    """3 x (conv3x3 -> [BN] -> ReLU -> maxpool2) followed by a 2-layer classifier."""
    def __init__(self, bn=False, dropout=0.0):
        super().__init__()
        def block(c_in, c_out):
            layers = [nn.Conv2d(c_in, c_out, 3, padding=1, bias=not bn)]
            if bn:
                layers.append(nn.BatchNorm2d(c_out))
            return layers + [nn.ReLU(inplace=True), nn.MaxPool2d(2)]
        self.features = nn.Sequential(*block(3, 32), *block(32, 64), *block(64, 128))  # 32->16->8->4
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(dropout), nn.Linear(128 * 4 * 4, 256), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(256, 10))

    def forward(self, x):
        return self.classifier(self.features(x))

print(f'SmallCNN: {count_params(SmallCNN()):,} parameters (with BN: {count_params(SmallCNN(bn=True)):,})')

OPTIMIZERS_2C = {
    'SGD': lambda ps, lr: MySGD(ps, lr=lr),
    'SGD + momentum': lambda ps, lr: MySGD(ps, lr=lr, momentum=0.9),
    'Adam': lambda ps, lr: MyAdam(ps, lr=lr),
}
LRS_2C = [1e-3, 1e-2, 1e-1]
EPOCHS_2C = 2 if QUICK else 20

def run_task2c():
    results = []
    for opt_name, make_opt in OPTIMIZERS_2C.items():
        for lr in LRS_2C:
            print(f'{opt_name}, lr = {lr:g}')
            set_seed()                                   # same initial weights and batch order
            model = SmallCNN().to(device)
            hist = train_model(model, make_opt(model.parameters(), lr), X_train, y_train, EPOCHS_2C,
                               batch_size=128, X_val=X_val, y_val=y_val, amp=False, log_every=5)
            results.append({'optimizer': opt_name, 'lr': lr, **hist})
    return results

res_2c = cached('task2c', run_task2c)

In [ ]:
df_2c = pd.DataFrame([{'optimizer': r['optimizer'], 'lr': r['lr'],
                       'best val acc': 100 * max(r['val_acc']), 'final val acc': 100 * r['val_acc'][-1],
                       'final train loss': r['train_loss'][-1], 'diverged': r['diverged']} for r in res_2c])
best_2c = df_2c.loc[df_2c.groupby('optimizer', sort=False)['best val acc'].idxmax()]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for (name, color), marker in zip(zip(OPTIMIZERS_2C, PALETTE), ['o', 's', '^']):
    d = df_2c[df_2c.optimizer == name]
    axes[0].semilogx(d.lr, d['best val acc'], marker=marker, ms=8, color=color, label=name)
    r = next(r for r in res_2c if r['optimizer'] == name
             and r['lr'] == best_2c[best_2c.optimizer == name].lr.item())
    epochs = np.arange(1, len(r['train_loss']) + 1)
    axes[1].plot(epochs, r['train_loss'], marker=marker, ms=4, color=color, label=f'{name} (lr={r["lr"]:g})')
    axes[2].plot(epochs, 100 * np.array(r['val_acc']), marker=marker, ms=4, color=color,
                 label=f'{name} (lr={r["lr"]:g})')
axes[0].axhline(10, ls=':', color='#898781', lw=1.2)
axes[0].text(1.1e-3, 12, 'chance (10 %)', color='#52514e', fontsize=8)
axes[0].set(xlabel='learning rate (log scale)', ylabel='best validation accuracy (%)',
            title='Validation accuracy vs learning rate', xticks=LRS_2C)
axes[1].set(xlabel='epoch', ylabel='training loss', title='Training loss at the best learning rate')
axes[2].set(xlabel='epoch', ylabel='validation accuracy (%)', title='Validation accuracy at the best learning rate')
for a in axes:
    a.legend(fontsize=8)
plt.tight_layout()
plt.show()

table_2c = df_2c.pivot(index='optimizer', columns='lr', values='best val acc').loc[list(OPTIMIZERS_2C)]
table_2c.columns = [f'lr={c:g}' for c in table_2c.columns]
print('Best validation accuracy (%) over the 20 epochs:')
display(table_2c.round(2))
print('Final-epoch validation accuracy (%):')
display(df_2c.pivot(index='optimizer', columns='lr', values='final val acc').loc[list(OPTIMIZERS_2C)].round(2))
if df_2c.diverged.any():
    print('Diverged runs (loss became NaN/inf):', df_2c[df_2c.diverged][['optimizer', 'lr']].values.tolist())

**Interpretation (sensitivity).**
* **Plain SGD** is the least sensitive to too-large steps but needs a *large* learning rate. At $10^{-3}$ it has barely started to learn after 20 epochs (‹xx %›), and its accuracy increases monotonically up to $10^{-1}$ (‹xx %›).
* **SGD with momentum** behaves like SGD with an effective learning rate of $\eta/(1-\mu) = 10\eta$. Its curve is therefore shifted about one decade to the left: ‹best at $10^{-2}$, xx %›, while $10^{-1}$ (effective step 1.0) ‹is unstable / much worse›.
* **Adam** works best at ‹$10^{-3}$ (xx %)›, which is the usual default. It is the most fragile at large learning rates: at ‹$10^{-1}$› the per-parameter normalised steps are too large for every weight at once, and the network ‹stalls at chance level / diverges›.
* At its best learning rate, ‹Adam› reduces the training loss fastest in the first epochs, because its per-parameter scaling copes with the very different gradient scales of the conv and FC layers. The best accuracies of the three methods are within ‹x› points of each other, so a well-tuned learning rate matters more than the choice of optimiser. The training loss keeps falling while the validation accuracy saturates, which shows overfitting (no regularisation is used here). The optimal learning rate differs by about two orders of magnitude between plain SGD and Adam, so learning rates cannot be transferred between optimisers: each one has to be tuned separately.

---
## Task 3 — Regularisation and residual connections

### 3(a) Regularisation on 5 000 training images
The model is `SmallCNN` **with batch normalisation**. It is trained on the first 5 000 images of the (already shuffled) 45k split and evaluated on the full 5k validation set. Every configuration uses SGD with momentum 0.9, learning rate 0.05 with cosine decay, batch size 128 and 80 epochs, and is repeated with **3 seeds** (mean ± std reported). Each configuration adds one technique to the baseline:

| configuration | change |
|---|---|
| baseline | — |
| weight decay | L2 penalty $5\times10^{-3}$ (larger than the usual $5\times10^{-4}$ because 80 epochs on 5k images give only about 3 100 updates) |
| dropout | $p = 0.5$ before both FC layers |
| flip + crop | random horizontal flip + random 32×32 crop after 4-pixel zero padding |
| cutout | one random 16×16 square set to zero (our implementation) |
| mixup | $\lambda \sim \mathrm{Beta}(0.4, 0.4)$, inputs and losses mixed (our implementation) |
| all combined | all of the above together |

Train accuracy is measured on the **clean** training images in eval mode, i.e. without augmentation, dropout or mixup. The gap (train − validation) is then a fair measure of overfitting.

In [ ]:
N_SMALL = 5000
X_small, y_small = X_train[:N_SMALL], y_train[:N_SMALL]
print('images per class in the 5k subset:', torch.bincount(y_small, minlength=10).tolist())

REG_CONFIGS = {
    'baseline':     dict(),
    'weight decay': dict(weight_decay=5e-3),
    'dropout':      dict(dropout=0.5),
    'flip + crop':  dict(augment=['flip', 'crop']),
    'cutout':       dict(augment=['cutout']),
    'mixup':        dict(mixup_alpha=0.4),
    'all combined': dict(weight_decay=5e-3, dropout=0.5, augment=['flip', 'crop', 'cutout'], mixup_alpha=0.4),
}
EPOCHS_3A = 3 if QUICK else 80
SEEDS_3A = [0] if QUICK else [0, 1, 2]

def run_task3a():
    out = {}
    for name, cfg in REG_CONFIGS.items():
        out[name] = []
        for seed in SEEDS_3A:
            print(f'{name} (seed {seed})')
            set_seed(seed)
            model = SmallCNN(bn=True, dropout=cfg.get('dropout', 0.0)).to(device)
            opt = MySGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=cfg.get('weight_decay', 0.0))
            out[name].append(train_model(
                model, opt, X_small, y_small, EPOCHS_3A, batch_size=128, lr_schedule=cosine_schedule(0.05),
                augment=make_augment(cfg.get('augment')), mixup_alpha=cfg.get('mixup_alpha', 0.0),
                X_val=X_val, y_val=y_val, eval_train=True, amp=USE_AMP, log_every=40))
    return out

res_3a = cached('task3a', run_task3a)

In [ ]:
rows, summary_3a = [], {}
for name, runs in res_3a.items():
    tr = 100 * np.array([r['train_eval_acc'][-1] for r in runs])
    va = 100 * np.array([r['val_acc'][-1] for r in runs])
    best = 100 * np.array([max(r['val_acc']) for r in runs])
    gap = tr - va
    summary_3a[name] = dict(train=tr.mean(), val=va.mean(), gap=gap.mean(), val_std=va.std(), gap_std=gap.std())
    rows.append({'configuration': name, 'train acc (%)': f'{tr.mean():.2f} ± {tr.std():.2f}',
                 'val acc (%)': f'{va.mean():.2f} ± {va.std():.2f}',
                 'gap (pp)': f'{gap.mean():.2f} ± {gap.std():.2f}',
                 'best-epoch val acc (%)': f'{best.mean():.2f}'})
print(f'Final epoch, mean ± std over {len(SEEDS_3A)} seed(s):')
display(pd.DataFrame(rows).set_index('configuration'))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for (name, runs), color in zip(res_3a.items(), PALETTE):
    lw = 2.8 if name in ('baseline', 'all combined') else 1.6
    epochs = np.arange(1, len(runs[0]['val_acc']) + 1)
    axes[0].plot(epochs, 100 * np.mean([r['train_eval_acc'] for r in runs], 0), color=color, lw=lw, label=name)
    axes[1].plot(epochs, 100 * np.mean([r['val_acc'] for r in runs], 0), color=color, lw=lw, label=name)
    axes[2].plot(epochs, np.mean([r['val_loss'] for r in runs], 0), color=color, lw=lw, label=name)
axes[0].set(xlabel='epoch', ylabel='accuracy (%)', title='Train accuracy (clean images)')
axes[1].set(xlabel='epoch', ylabel='accuracy (%)', title='Validation accuracy')
axes[2].set(xlabel='epoch', ylabel='cross-entropy', title='Validation loss')
axes[1].legend(fontsize=8, loc='lower right')
plt.suptitle('SmallCNN + BN trained on 5k images (curves averaged over seeds)', y=1.02)
plt.tight_layout()
plt.show()

names = list(summary_3a)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].barh(names, [summary_3a[n]['val'] for n in names], xerr=[summary_3a[n]['val_std'] for n in names],
             color=PALETTE[0], height=0.6)
axes[0].set(xlabel='validation accuracy (%)', title='Validation accuracy (final epoch)')
axes[1].barh(names, [summary_3a[n]['gap'] for n in names], xerr=[summary_3a[n]['gap_std'] for n in names],
             color=PALETTE[1], height=0.6)
axes[1].set(xlabel='train − validation accuracy (pp)', title='Generalisation gap')
for a in axes:
    a.invert_yaxis()
    a.grid(axis='y', visible=False)
    for patch in a.patches:
        a.text(patch.get_width() + 0.3, patch.get_y() + patch.get_height() / 2, f'{patch.get_width():.1f}',
               va='center', fontsize=8, color='#52514e')
plt.tight_layout()
plt.show()

**Interpretation.**
* **Baseline:** with only 5 000 images, the 620k-parameter CNN memorises the training set (train accuracy ‹≈100 %›) and reaches ‹xx %› on validation, a gap of about ‹xx› points. Its validation loss rises again after about epoch ‹k›, while its validation accuracy stays flat: the network becomes over-confident on the images it gets wrong. This is classic overfitting.
* **Weight decay** and **dropout** constrain the model itself. ‹They reduce the gap by x / y points and change the validation accuracy by …›. Dropout acts only on the FC head, which holds most of the parameters. With batch normalisation, weight decay on the conv layers mostly increases the effective learning rate, so its benefit comes mainly from the FC layers.
* **Flip + crop** adds data that is genuinely new and label-preserving (a cat shifted by a few pixels or mirrored is still a cat). It is ‹the most effective single technique (+xx points)›, because it encodes invariances that the true data distribution has.
* **Cutout** forces the network to use several parts of the object instead of a single discriminative patch; ‹effect›. **Mixup** trains on convex combinations with soft labels. It lowers the clean training accuracy less than the training loss suggests, and makes the decision boundaries more linear between classes; ‹effect›.
* **All combined** gives ‹the smallest gap (xx points) and the best / not the best validation accuracy (xx %)›. ‹If it is not the best: the combined regularisers are so strong that the model underfits within 80 epochs, i.e. the effects do not simply add up and the strength of each technique would have to be retuned.›

Overall, **data-side** regularisation (augmentation) is more effective on small datasets than **model-side** regularisation (weight decay, dropout). Augmentation effectively enlarges the training set, whereas the model-side methods only limit how well the network can fit the few images it has.

### 3(b) Plain vs residual networks of depth $6n+2$
The CIFAR architecture of He et al. (2016): conv3×3 (16 channels), then 3 stages of $n$ blocks (16/32/64 channels, the first block of stages 2 and 3 uses stride 2), global average pooling and an FC layer. Each block has two conv3×3–BN layers. Depth $= 1 + 6n + 1$, so $n = 1, 3, 9$ gives depths 8, 20 and 56. Conv layers use Kaiming normal initialisation; BN starts with $\gamma = 1, \beta = 0$.

**The only difference is the shortcut.** In the residual net, a block outputs $\mathrm{ReLU}(F(x) + x)$; in the plain net it outputs $\mathrm{ReLU}(F(x))$. When the shape changes, the residual net uses the **parameter-free option A shortcut** (spatial subsampling plus zero-padding of the new channels). Both families therefore have *exactly* the same layers and parameter counts.

Training: SGD with momentum 0.9, learning rate 0.1 (1-epoch warm-up, then cosine decay), weight decay $10^{-4}$, batch size 128, flip + crop augmentation, 30 epochs on the 45k split.

In [ ]:
class BasicBlock(nn.Module):
    """conv3x3-BN-ReLU-conv3x3-BN (+ identity shortcut if residual) -> ReLU."""
    def __init__(self, c_in, c_out, stride, residual):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.residual, self.stride, self.extra = residual, stride, c_out - c_in

    def shortcut(self, x):
        """Option A: parameter-free -- subsample spatially and zero-pad the extra channels."""
        if self.stride != 1 or self.extra:
            x = x[:, :, ::self.stride, ::self.stride]
            x = F.pad(x, (0, 0, 0, 0, self.extra // 2, self.extra - self.extra // 2))
        return x

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.residual:
            out = out + self.shortcut(x)          # the ONLY difference between the two families
        return F.relu(out)


class CifarNet(nn.Module):
    """Depth 6n+2: conv -> 3 stages x n BasicBlocks (16/32/64 ch) -> global avg pool -> FC."""
    def __init__(self, depth, residual=True, num_classes=10):
        super().__init__()
        assert (depth - 2) % 6 == 0, 'depth must be 6n + 2'
        n = (depth - 2) // 6
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16), nn.ReLU(inplace=True))
        blocks, c_in = [], 16
        for stage, c_out in enumerate([16, 32, 64]):
            for i in range(n):
                blocks.append(BasicBlock(c_in, c_out, 2 if (stage > 0 and i == 0) else 1, residual))
                c_in = c_out
        self.blocks = nn.Sequential(*blocks)
        self.fc = nn.Linear(64, num_classes)
        for m in self.modules():                  # Kaiming initialisation
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.blocks(self.stem(x))
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)   # global average pooling
        return self.fc(x)

DEPTHS = [8, 20, 56]
for d in DEPTHS:
    n_conv = sum(isinstance(m, nn.Conv2d) for m in CifarNet(d).modules())
    print(f'depth {d:2d}: {n_conv} conv + 1 FC layers | plain {count_params(CifarNet(d, False)):,} params'
          f' | residual {count_params(CifarNet(d, True)):,} params')

In [ ]:
EPOCHS_3B = 2 if QUICK else 30

def run_task3b():
    out = {}
    for residual in [False, True]:
        for depth in DEPTHS:
            name = f'{"ResNet" if residual else "Plain"}-{depth}'
            print(name)
            set_seed()
            model = CifarNet(depth, residual).to(device)
            opt = MySGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
            out[name] = train_model(model, opt, X_train, y_train, EPOCHS_3B, batch_size=128,
                                    lr_schedule=cosine_schedule(0.1, warmup_frac=1 / EPOCHS_3B),
                                    augment=make_augment(['flip', 'crop']), X_val=X_val, y_val=y_val,
                                    amp=USE_AMP, log_every=5)
    return out

res_3b = cached('task3b', run_task3b)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
for depth, color in zip(DEPTHS, PALETTE):
    for fam, ls, marker in (('Plain', '--', 's'), ('ResNet', '-', 'o')):
        h = res_3b[f'{fam}-{depth}']
        ep = np.arange(1, len(h['train_loss']) + 1)
        axes[0].plot(ep, h['train_loss'], ls=ls, marker=marker, ms=3, color=color, label=f'{fam}-{depth}')
        axes[1].plot(ep, 100 * np.array(h['val_acc']), ls=ls, marker=marker, ms=3, color=color,
                     label=f'{fam}-{depth}')
axes[0].set(xlabel='epoch', ylabel='training loss (log scale)', yscale='log',
            title='Training loss: plain (dashed) vs residual (solid)')
axes[1].set(xlabel='epoch', ylabel='validation accuracy (%)', title='Validation accuracy')
axes[0].legend(ncol=2, fontsize=8)
axes[1].legend(ncol=2, fontsize=8, loc='lower right')
plt.tight_layout()
plt.show()

rows = []
for depth in DEPTHS:
    for fam in ('Plain', 'ResNet'):
        h = res_3b[f'{fam}-{depth}']
        rows.append({'network': f'{fam}-{depth}', 'final train loss': round(h['train_loss'][-1], 4),
                     'train loss @ epoch 5': round(h['train_loss'][min(4, len(h['train_loss']) - 1)], 4),
                     'final val acc (%)': round(100 * h['val_acc'][-1], 2),
                     'time (min)': round(h['time'][-1] / 60, 1)})
display(pd.DataFrame(rows).set_index('network'))

To see *why* depth hurts the plain network, the next cell plots the gradient norm of the first conv layer of every block at initialisation, for both 56-layer networks (one batch of 256 images).

In [ ]:
def block_grad_norms(depth, residual):
    set_seed()
    model = CifarNet(depth, residual).to(device).train()
    F.cross_entropy(model(X_train[:256]), y_train[:256]).backward()
    return [blk.conv1.weight.grad.norm().item() for blk in model.blocks]

fig, ax = plt.subplots(figsize=(9, 4))
for (fam, residual), ls, marker in ((('Plain-56', False), '--', 's'), (('ResNet-56', True), '-', 'o')):
    g = block_grad_norms(56, residual)
    ax.semilogy(np.arange(1, len(g) + 1), g, ls=ls, marker=marker, ms=4, label=fam)
ax.set(xlabel='block (1 = closest to the input)', ylabel=r'$\|\partial L/\partial W_{conv1}\|$ (log scale)',
       title='Gradient norm per block at initialisation (depth 56)')
ax.legend()
plt.tight_layout()
plt.show()

**Interpretation.**
* **Plain networks get *worse* with depth.** Plain-56 has a ‹clearly higher› training loss than Plain-20 throughout training (‹x› vs ‹y› at the end), and Plain-20 ‹is close to / worse than› Plain-8. This is the **degradation problem** (He et al., 2016). It is an *optimisation* failure, not overfitting, because it shows up in the *training* loss. A 56-layer plain net could in principle represent the 20-layer solution, by setting the extra layers to the identity, but SGD does not find such a solution. Batch normalisation keeps the activations well scaled, so this is not plain vanishing of the signal. The gradient-norm plot shows instead that in the plain net the gradient magnitude varies by ‹orders of magnitude› across the 27 blocks: the gradients of very deep BN networks become erratic and badly conditioned ("shattered").
* **Residual networks get *better* with depth.** ResNet-56 < ResNet-20 < ResNet-8 in training loss (‹values›), and each ResNet is below its plain counterpart. The difference is largest at depth 56. Each block only has to learn a residual $F(x) = H(x) - x$, which is easy when the identity is close to optimal. During backpropagation the identity path gives $\partial x_L/\partial x_\ell = I + \partial(\sum_k F_k)/\partial x_\ell$, so the gradient reaches the early blocks directly and does not pass through every weight layer. Consistently with this, the residual curve in the gradient-norm plot is ‹much flatter›.
* At depth 8 the shortcut makes little difference, since there are only 3 blocks. The validation accuracies follow the training losses (‹ResNet-56 best, xx %›), which confirms that the shortcut helps generalisation by making deep networks *trainable*.

---
## Task 4 — Best CNN with at most 500k parameters and 20 epochs (optional)

**Design.** A compact residual network, *ResNet-12-w32*:
* stem conv3×3 (32 channels) → 3 stages of BasicBlocks with widths **32 / 64 / 128** and **2 / 2 / 1** blocks (stride 2 at the start of stages 2 and 3), then global average pooling and an FC layer. 1×1 conv projections are used where the shape changes (option B).
* The width is spent at the high resolutions (32 channels at 32×32), where most of the useful computation for CIFAR-10 happens, while the channel count of the last stage, which dominates the parameter count, is kept at 128. Total: about **401k parameters**.
* Training recipe: SGD with momentum 0.9 (our `MySGD`), batch size 128, linear warm-up over 10 % of the steps followed by cosine decay, weight decay $5\times10^{-4}$, label smoothing 0.1, flip + crop + cutout 8×8, fp16 mixed precision, **20 epochs**.

**Tuning on the validation split.** (1) The peak learning rate is chosen from $\{0.05, 0.1, 0.2\}$. (2) At that learning rate, six one-factor-at-a-time ablations are run: no shortcut, no augmentation, no cutout, no weight decay, no label smoothing, and Adam instead of SGD. (3) The run with the highest final validation accuracy becomes the final model. It is evaluated on the **test set once**.

In [ ]:
class ResBlock(nn.Module):
    """BasicBlock with an optional shortcut; a 1x1 conv + BN projection is used when the shape changes."""
    def __init__(self, c_in, c_out, stride, shortcut=True):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.use_shortcut = shortcut
        self.proj = None
        if shortcut and (stride != 1 or c_in != c_out):
            self.proj = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.use_shortcut:
            out = out + (x if self.proj is None else self.proj(x))
        return F.relu(out)


class CompactResNet(nn.Module):
    def __init__(self, widths=(32, 64, 128), blocks=(2, 2, 1), shortcut=True, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, widths[0], 3, 1, 1, bias=False),
                                  nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True))
        layers, c_in = [], widths[0]
        for stage, (c_out, n) in enumerate(zip(widths, blocks)):
            for i in range(n):
                layers.append(ResBlock(c_in, c_out, 2 if (stage > 0 and i == 0) else 1, shortcut))
                c_in = c_out
        self.layers = nn.Sequential(*layers)
        self.fc = nn.Linear(c_in, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        x = self.layers(self.stem(x))
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

n_params = count_params(CompactResNet())
print(f'CompactResNet: {n_params:,} parameters '
      f'(without shortcuts: {count_params(CompactResNet(shortcut=False)):,})')
assert n_params <= 500_000, 'parameter budget exceeded'

In [ ]:
EPOCHS_4 = 2 if QUICK else 20
assert EPOCHS_4 <= 20
BASE_4 = dict(lr=0.1, optimizer='sgd', weight_decay=5e-4, augment=['flip', 'crop', 'cutout8'],
              label_smoothing=0.1, shortcut=True)

def train_task4(cfg, tag):
    print(f'--- {tag}: {cfg}')
    set_seed()
    model = CompactResNet(shortcut=cfg['shortcut']).to(device)
    if cfg['optimizer'] == 'sgd':
        opt = MySGD(model.parameters(), lr=cfg['lr'], momentum=0.9, weight_decay=cfg['weight_decay'])
    else:
        opt = MyAdam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    hist = train_model(model, opt, X_train, y_train, EPOCHS_4, batch_size=128,
                       lr_schedule=cosine_schedule(cfg['lr'], warmup_frac=0.1),
                       augment=make_augment(cfg['augment']), label_smoothing=cfg['label_smoothing'],
                       X_val=X_val, y_val=y_val, amp=USE_AMP, log_every=5)
    torch.save(model.state_dict(), os.path.join(RESULTS_DIR, f'task4_{tag}.pt'))
    return {'tag': tag, 'config': cfg, 'params': count_params(model), **hist}

# Step 1: tune the peak learning rate of the full recipe on the validation split.
runs_4 = {}
for lr in [0.05, 0.1, 0.2]:
    cfg = dict(BASE_4, lr=lr)
    runs_4[f'full, lr={lr:g}'] = cached(f'task4_full_lr{lr:g}', lambda: train_task4(cfg, f'full_lr{lr:g}'))
best_lr = max([0.05, 0.1, 0.2], key=lambda lr: runs_4[f'full, lr={lr:g}']['val_acc'][-1])
print(f'\nBest peak learning rate on the validation split: {best_lr:g}')

In [ ]:
# Step 2: ablations, one design choice changed at a time (relative to the full recipe at best_lr).
REF_4 = dict(BASE_4, lr=best_lr)
ABLATIONS_4 = {
    'no shortcut':           dict(shortcut=False),
    'no augmentation':       dict(augment=[]),
    'no cutout (flip+crop)': dict(augment=['flip', 'crop']),
    'no weight decay':       dict(weight_decay=0.0),
    'no label smoothing':    dict(label_smoothing=0.0),
    'Adam instead of SGD':   dict(optimizer='adam', lr=2e-3),
}
for name, change in ABLATIONS_4.items():
    tag = 'abl_' + name.split(' (')[0].replace(' ', '_') + f'_lr{best_lr:g}'
    cfg = dict(REF_4, **change)
    runs_4[name] = cached(f'task4_{tag}', lambda: train_task4(cfg, tag))

In [ ]:
ref_name = f'full, lr={best_lr:g}'
ref_acc = 100 * runs_4[ref_name]['val_acc'][-1]
rows = []
for name, r in runs_4.items():
    acc = 100 * r['val_acc'][-1]
    rows.append({'run': name + ('  ← reference' if name == ref_name else ''), 'params': f'{r["params"]:,}',
                 'final val acc (%)': round(acc, 2),
                 'Δ vs reference (pp)': '—' if name == ref_name or name.startswith('full') else f'{acc - ref_acc:+.2f}',
                 'final train loss': round(r['train_loss'][-1], 4),
                 'time (min)': round(r['time'][-1] / 60, 1)})
display(pd.DataFrame(rows).set_index('run'))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
abl_names = [ref_name] + list(ABLATIONS_4)
for name, color in zip(abl_names, PALETTE):
    r = runs_4[name]
    ep = np.arange(1, len(r['val_acc']) + 1)
    lw = 3 if name == ref_name else 1.6
    axes[0].plot(ep, 100 * np.array(r['val_acc']), color=color, lw=lw, label=name)
    axes[1].plot(ep, r['train_loss'], color=color, lw=lw, label=name)
axes[0].set(xlabel='epoch', ylabel='validation accuracy (%)', title='Ablations: validation accuracy', ylim=(50, 100))
axes[1].set(xlabel='epoch', ylabel='training loss', title='Ablations: training loss (incl. label smoothing)')
axes[0].legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.show()

deltas = {n: 100 * runs_4[n]['val_acc'][-1] - ref_acc for n in ABLATIONS_4}
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.barh(list(deltas), list(deltas.values()), color=[PALETTE[7] if v < 0 else PALETTE[2] for v in deltas.values()],
        height=0.6)
ax.axvline(0, color='#52514e', lw=1)
for i, v in enumerate(deltas.values()):
    ax.text(v + (0.05 if v >= 0 else -0.05), i, f'{v:+.2f}', va='center', ha='left' if v >= 0 else 'right',
            fontsize=8, color='#52514e')
ax.invert_yaxis()
ax.grid(axis='y', visible=False)
ax.set(xlabel='change in validation accuracy vs full recipe (pp)', title='Effect of removing / changing one design choice')
plt.tight_layout()
plt.show()

**Final model and the single test-set evaluation.** The final model is the run with the highest final validation accuracy among all runs above, so it is selected on the validation split only. Its saved weights are loaded and evaluated **once** on the 10 000 CIFAR-10 test images. The result is cached, so re-running the notebook does not evaluate the test set again.

In [ ]:
final_name = max(runs_4, key=lambda n: runs_4[n]['val_acc'][-1])
final_run = runs_4[final_name]
print(f'Selected on validation: "{final_name}"  (val acc {100 * final_run["val_acc"][-1]:.2f} %, '
      f'{final_run["params"]:,} parameters, {EPOCHS_4} epochs)')

def evaluate_test_once(run):
    path = os.path.join(RESULTS_DIR, 'task4_TEST_RESULT.json')
    if os.path.exists(path):                       # the test set has already been used once
        print('[cache] test result loaded (the test set is not evaluated again)')
        with open(path) as f:
            return json.load(f)
    model = CompactResNet(shortcut=run['config']['shortcut']).to(device)
    model.load_state_dict(torch.load(os.path.join(RESULTS_DIR, f'task4_{run["tag"]}.pt'), map_location=device))
    model.eval()
    X_test = normalize(torch.from_numpy(cifar_test.data).permute(0, 3, 1, 2).float().div(255)).to(device)
    y_test = torch.tensor(cifar_test.targets, device=device)
    with torch.no_grad(), autocast(USE_AMP):
        preds = torch.cat([model(X_test[i:i + 1000]).argmax(1) for i in range(0, len(X_test), 1000)])
    confusion = torch.bincount(y_test * 10 + preds, minlength=100).view(10, 10)
    result = {'model': run['tag'], 'test_acc': (preds == y_test).float().mean().item(),
              'confusion': confusion.tolist()}
    with open(path, 'w') as f:
        json.dump(result, f)
    return result

test_result = evaluate_test_once(final_run)
print(f'TEST ACCURACY (evaluated once): {100 * test_result["test_acc"]:.2f} %')

cm = np.array(test_result['confusion'])
cm_pct = 100 * cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(7.5, 6.3))
im = ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=100)
for i in range(10):
    for j in range(10):
        ax.text(j, i, f'{cm_pct[i, j]:.0f}', ha='center', va='center', fontsize=8,
                color='white' if cm_pct[i, j] > 60 else '#0b0b0b')
ax.set_xticks(range(10)); ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticks(range(10)); ax.set_yticklabels(CLASSES)
ax.set(xlabel='predicted class', ylabel='true class', title='Test-set confusion matrix (% of each true class)')
ax.grid(False)
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()
print('Per-class test accuracy (%):', {c: round(cm_pct[i, i], 1) for i, c in enumerate(CLASSES)})

**Interpretation.**
* **Learning rate:** ‹0.05 / 0.1 / 0.2 gave xx / xx / xx %›. Thanks to batch normalisation and the warm-up, the recipe is fairly robust to the peak learning rate, and ‹best_lr› was kept.
* **Ablations** (validation accuracy relative to the full recipe):
  * *Augmentation* is ‹the most important choice (−xx points without it)›. Without it the network fits the 45k images within 20 epochs and starts to overfit; with flip + crop it sees a different version of every image in each epoch.
  * *Shortcut:* removing it costs ‹xx points›. Even at about 12 layers, the identity paths speed up optimisation, which matters with only 20 epochs.
  * *Weight decay* ‹−xx›: with BN, weight decay keeps the weight norms small, which keeps the effective learning rate high, and it regularises the FC layer.
  * *Cutout* ‹±xx› and *label smoothing* ‹±xx› are smaller refinements.
  * *Adam* instead of SGD ‹−xx›: in line with Task 2(c) and with the literature, a well-tuned SGD with momentum generalises better than Adam with L2 weight decay on CIFAR ResNets.
* **Final result:** the final model (‹name›, ‹401k› parameters, 20 epochs) reaches **‹xx.x %› validation** and **‹xx.x %› test accuracy**. The test accuracy is ‹close to› the validation accuracy, which shows that the choices made on the validation split did not overfit it. The confusion matrix shows that most errors are between visually similar classes (‹cat ↔ dog, automobile ↔ truck, bird/deer›), while vehicles such as ‹ship and automobile› are recognised best.

---
## Summary

In [ ]:
print('Key results of this notebook')
print(f'  1(c) from-scratch MLP, final validation accuracy : {100 * hist_mlp["val_acc"][-1]:.2f} %')
for _, r in best_2c.iterrows():
    print(f'  2(c) {r.optimizer:15s} best lr {r.lr:<6g}           : {r["best val acc"]:.2f} % validation')
for name, s in summary_3a.items():
    print(f'  3(a) {name:15s} val {s["val"]:.2f} %, gap {s["gap"]:.2f} pp')
for depth in DEPTHS:
    p, r = res_3b[f'Plain-{depth}'], res_3b[f'ResNet-{depth}']
    print(f'  3(b) depth {depth:2d}: final train loss plain {p["train_loss"][-1]:.4f} | residual {r["train_loss"][-1]:.4f}')
print(f'  4    final model "{final_name}": validation {100 * final_run["val_acc"][-1]:.2f} %, '
      f'TEST {100 * test_result["test_acc"]:.2f} %')

**Conclusions.**
1. **Backpropagation** is the chain rule applied layer by layer. A hand-written backward pass agrees with autograd to machine precision and with finite differences to the accuracy of the numerical derivative. Gradient flow through depth depends on the activation derivative and on the initialisation: sigmoid gradients vanish geometrically, while ReLU with He initialisation preserves their scale.
2. **Optimisers:** momentum fixes the zig-zag of SGD in ill-conditioned valleys, and Adam rescales each coordinate. In practice the learning rate is the most important hyper-parameter, and its good range shifts by orders of magnitude between optimisers.
3. **Regularisation:** with little data, label-preserving augmentation is the most effective regulariser, and stacking regularisers needs retuning to avoid underfitting.
4. **Residual connections** solve the degradation problem: deeper plain networks train worse, while deeper ResNets train better.
5. **Combining these lessons** (residual design, augmentation, SGD with momentum, a cosine schedule with warm-up, weight decay) gives ‹xx %› test accuracy with about 401k parameters in 20 epochs.

### References
* K. He, X. Zhang, S. Ren, J. Sun. *Delving deep into rectifiers* (He initialisation), ICCV 2015.
* K. He, X. Zhang, S. Ren, J. Sun. *Deep residual learning for image recognition*, CVPR 2016.
* X. Glorot, Y. Bengio. *Understanding the difficulty of training deep feedforward neural networks*, AISTATS 2010.
* D. P. Kingma, J. Ba. *Adam: a method for stochastic optimization*, ICLR 2015.
* S. Ioffe, C. Szegedy. *Batch normalization*, ICML 2015.
* N. Srivastava et al. *Dropout*, JMLR 2014.
* T. DeVries, G. W. Taylor. *Improved regularization of CNNs with Cutout*, arXiv:1708.04552, 2017.
* H. Zhang, M. Cisse, Y. Dauphin, D. Lopez-Paz. *mixup: beyond empirical risk minimization*, ICLR 2018.
* I. Loshchilov, F. Hutter. *SGDR: stochastic gradient descent with warm restarts*, ICLR 2017.